# Loading the datasets

In [ ]:
#Importar librerias
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import ptitprince as pt
pd.options.display.max_rows = 999
import warnings
warnings.filterwarnings("ignore")
import sys
sys.dont_write_bytecode = True

In [ ]:
# Replace
path = r"C:\Users\Mosqu\universidadean.edu.co\MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes\REPLACE-BG Dataset-79f6bdc8-3c51-4736-a39f-c4c0f71d45e5\Data Tables\HDeviceCGM.txt"
def Loading_File(file_path):
    MasterDF = pd.read_csv(file_path, sep='|', header=0, low_memory = False)
    MasterDF = MasterDF[MasterDF['RecordType'] == 'CGM']
    #Creating a date time column
    MasterDF['Today'] = datetime.today().date()
    MasterDF['Date'] = MasterDF['Today'] + pd.to_timedelta(MasterDF['DeviceDtTmDaysFromEnroll'], unit='d')
    MasterDF['DeviceTm'] = MasterDF.DeviceTm.astype('str')
    MasterDF['DeviceTm'] = MasterDF['DeviceTm'].str[:-2]+ '00'
    MasterDF['DateTime'] = MasterDF.Date.astype('str')+ ' '+ MasterDF.DeviceTm
    MasterDF['DateTime'] = pd.to_datetime(MasterDF.DateTime, format='%Y-%m-%d %H:%M:%S')
    #selecting just the columns for Giammarino's code to run
    MasterDF = MasterDF[['PtID','DateTime','GlucoseValue']]
    MasterDF = MasterDF.rename(columns={'DateTime':'ts','PtID':'id','GlucoseValue':'gl'})
    MasterDF= MasterDF.reset_index(drop=True)
    MasterDF = MasterDF.drop_duplicates(subset=['ts','id'])
    data = MasterDF.pivot(index='ts', columns=['id'], values=['gl'])
    data.columns = data.columns.get_level_values(level='id')
    return data
Replace = Loading_File(path)

In [ ]:
# AIDE
path = r"C:\Users\Mosqu\universidadean.edu.co\MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes\AIDE T1D\Data Tables\AIDEDeviceCGM.txt"
def Loading_File(file_path):
    MasterDF = pd.read_csv(file_path, sep='|', header=0, low_memory = False)
    MasterDF = MasterDF[MasterDF['RecordType'] == 'CGM']
    #Creating a date time column
    MasterDF['DateTime'] = MasterDF['DataDtTm']
    MasterDF['DateTime'] = pd.to_datetime(MasterDF.DateTime, errors='coerce', infer_datetime_format=True)
    #selecting just the columns for Giammarino's code to run
    MasterDF = MasterDF[['PtID','DateTime','GlucValue']]
    MasterDF = MasterDF.rename(columns={'DateTime':'ts','PtID':'id','GlucValue':'gl'})
    MasterDF= MasterDF.reset_index(drop=True)
    MasterDF = MasterDF.drop_duplicates(subset=['ts','id'])
    data = MasterDF.pivot(index='ts', columns=['id'], values=['gl'])
    data.columns = data.columns.get_level_values(level='id')
    return data
AIDE = Loading_File(path)

In [ ]:
# Shanghai
from pathlib import Path
T1D = r"C:\Users\Mosqu\universidadean.edu.co\MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes\Dataset Shangai\Shanghai_T1DM"
T2D = r"C:\Users\Mosqu\universidadean.edu.co\MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes\Dataset Shangai\Shanghai_T2DM"
def Loading_File(file_path):
    files = list(Path(file_path).glob('*.xls*'))
    folder_dfs = pd.DataFrame()
    # print(files[0])
    for file in files:
        try:
            pti = str(file)[124:128]
            df_temp = pd.read_excel(str(file))
            df_temp = df_temp.iloc[:,0:2]
            df_temp = df_temp.rename(columns={df_temp.columns[0]: 'DateTime'})
            df_temp = df_temp.rename(columns={df_temp.columns[1]: 'GlucValue'})
            df_temp['PtID'] = pti
            df_temp['DateTime'] = pd.to_datetime(df_temp.DateTime, errors='coerce', infer_datetime_format=True)
            df_temp = df_temp[['PtID','DateTime','GlucValue']]
            df_temp = df_temp.rename(columns={'DateTime':'ts','PtID':'id','GlucValue':'gl'})
            df_temp= df_temp.reset_index(drop=True)
            df_temp = df_temp.drop_duplicates(subset=['ts','id'])
        except Exception as e:
            print(f"Error al leer {file.name}: {e}")
        folder_dfs = pd.concat([folder_dfs, df_temp])
    return folder_dfs
Shanghai_T1D = Loading_File(T1D)
Shanghai_T2D = Loading_File(T2D)
Shanghai = pd.concat([Shanghai_T1D, Shanghai_T2D])
#pivoting the dataset
data = Shanghai.pivot(index='ts', columns=['id'], values=['gl'])
data.columns = data.columns.get_level_values(level='id')
Shanghai = data


# Transform them into Sequences

In [26]:
from src.utils import get_labelled_sequences
# minimum percentage of time that the patient must have worn the device over a given week
time_worn_threshold = 0.7

# glucose threshold below which we detect the onset of hypoglycemia, in mg/dL
glucose_threshold = 54

# minimum length of a hypoglycemic event, in minutes
event_duration_threshold = 15

sequences = get_labelled_sequences(
    data=data,
    time_worn_threshold=time_worn_threshold,
    glucose_threshold=glucose_threshold,
    event_duration_threshold=event_duration_threshold,
)

ts
2020-01-17 18:57:32   NaN
2020-01-17 19:02:33   NaN
2020-01-17 19:07:33   NaN
2020-01-17 19:12:33   NaN
2020-01-17 19:17:33   NaN
                       ..
2020-01-30 03:38:05   NaN
2020-01-30 03:43:06   NaN
2020-01-30 03:48:06   NaN
2020-01-30 03:53:06   NaN
2020-01-30 03:58:05   NaN
Name: 4, Length: 2016, dtype: float64
ts
2020-01-30 04:03:05   NaN
2020-01-30 04:08:06   NaN
2020-01-30 04:13:06   NaN
2020-01-30 04:18:05   NaN
2020-01-30 04:23:05   NaN
                       ..
2020-02-04 04:28:13   NaN
2020-02-04 04:28:17   NaN
2020-02-04 04:33:13   NaN
2020-02-04 04:33:18   NaN
2020-02-04 04:38:13   NaN
Name: 4, Length: 2016, dtype: float64
ts
2020-02-04 04:38:17   NaN
2020-02-04 04:43:13   NaN
2020-02-04 04:43:17   NaN
2020-02-04 04:48:13   NaN
2020-02-04 04:48:17   NaN
                       ..
2020-02-07 20:13:25   NaN
2020-02-07 20:18:16   NaN
2020-02-07 20:18:25   NaN
2020-02-07 20:23:16   NaN
2020-02-07 20:23:25   NaN
Name: 4, Length: 2016, dtype: float64
ts
2020-02-07 20:28